In [20]:
# Download all HVFHV 2024 months directly from NYC TLC (official CloudFront CDN)
# and verify each file is a valid Parquet.

import os, subprocess, shlex
import pyarrow.parquet as pq

BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
YEAR = 2024
OUT_DIR = "/content/hvfhv_2024"
os.makedirs(OUT_DIR, exist_ok=True)

def download_month(mm: str):
    url = f"{BASE_URL}/fhvhv_tripdata_{YEAR}-{mm}.parquet"
    out = f"{OUT_DIR}/fhvhv_tripdata_{YEAR}-{mm}.parquet"
    print(f"↓ {YEAR}-{mm}  ->  {out}")
    # robust curl (follow redirects, retry, show progress)
    cmd = f"curl -L --fail --retry 3 --progress-bar {shlex.quote(url)} -o {shlex.quote(out)}"
    subprocess.check_call(cmd, shell=True)
    return out

def validate_parquet(path: str) -> bool:
    try:
        pq.ParquetFile(path)  # open footer & metadata without full read
        return True
    except Exception as e:
        print(f"✗ INVALID parquet: {os.path.basename(path)} -> {e}")
        return False

# 1) download all months (01..12)
downloaded = []
for mm in [f"{m:02d}" for m in range(1, 13)]:
    try:
        p = download_month(mm)
        downloaded.append(p)
    except subprocess.CalledProcessError as e:
        print(f"✗ DOWNLOAD FAILED for {YEAR}-{mm}: {e}")

# 2) validate all
print("\nValidating downloaded files…")
valid, invalid = [], []
for p in downloaded:
    (valid if validate_parquet(p) else invalid).append(p)

# 3) summary
import os
print("\nSummary:")
for p in sorted(valid):
    sz = os.path.getsize(p) / (1024**2)
    print(f"✓ {os.path.basename(p):35s}  {sz:8.2f} MB")
for p in sorted(invalid):
    print(f"✗ {os.path.basename(p)} (re-download manually)")

print(f"\nOK: {len(valid)}   BAD: {len(invalid)}   Folder: {OUT_DIR}")



↓ 2024-01  ->  /content/hvfhv_2024/fhvhv_tripdata_2024-01.parquet
↓ 2024-02  ->  /content/hvfhv_2024/fhvhv_tripdata_2024-02.parquet
↓ 2024-03  ->  /content/hvfhv_2024/fhvhv_tripdata_2024-03.parquet
↓ 2024-04  ->  /content/hvfhv_2024/fhvhv_tripdata_2024-04.parquet
↓ 2024-05  ->  /content/hvfhv_2024/fhvhv_tripdata_2024-05.parquet
↓ 2024-06  ->  /content/hvfhv_2024/fhvhv_tripdata_2024-06.parquet
↓ 2024-07  ->  /content/hvfhv_2024/fhvhv_tripdata_2024-07.parquet
↓ 2024-08  ->  /content/hvfhv_2024/fhvhv_tripdata_2024-08.parquet
↓ 2024-09  ->  /content/hvfhv_2024/fhvhv_tripdata_2024-09.parquet
↓ 2024-10  ->  /content/hvfhv_2024/fhvhv_tripdata_2024-10.parquet
↓ 2024-11  ->  /content/hvfhv_2024/fhvhv_tripdata_2024-11.parquet
↓ 2024-12  ->  /content/hvfhv_2024/fhvhv_tripdata_2024-12.parquet

Validating downloaded files…

Summary:
✓ fhvhv_tripdata_2024-01.parquet         450.86 MB
✓ fhvhv_tripdata_2024-02.parquet         441.05 MB
✓ fhvhv_tripdata_2024-03.parquet         484.41 MB
✓ fhvhv_tripdat

In [22]:
RAW_DIR = "/content/hvfhv_2024"
OUT_DIR = "/content/processed"
CLEAN_PATH = f"{OUT_DIR}/hvfhv_2024_clean.parquet"
ZONE_HOUR_PATH = f"{OUT_DIR}/zone_hour_2024.parquet"

import os
os.makedirs(OUT_DIR, exist_ok=True)


In [23]:
import duckdb
con = duckdb.connect()
con.execute("SET timezone='America/New_York'")

q_clean = f"""
WITH raw AS (
  SELECT
    CAST(pickup_datetime AS TIMESTAMP) AS pickup_datetime,
    CAST(PULocationID AS INTEGER)      AS PULocationID,
    CAST(DOLocationID AS INTEGER)      AS DOLocationID
  FROM read_parquet('{RAW_DIR}/fhvhv_tripdata_2024-*.parquet')
  WHERE hvfhs_license_num IN ('HV0003','HV0005')      -- Uber + Lyft only
    AND pickup_datetime IS NOT NULL
    AND PULocationID    IS NOT NULL
)
SELECT * FROM raw
"""

df_clean = con.execute(q_clean).fetch_df()
df_clean.to_parquet(CLEAN_PATH, index=False)
print(f"Clean file saved: {CLEAN_PATH}  ({len(df_clean):,} rows)")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Clean file saved: /content/processed/hvfhv_2024_clean.parquet  (239,470,448 rows)


In [24]:
q_zone_hour = """
WITH agg AS (
  SELECT
    PULocationID,
    date_trunc('hour', pickup_datetime) AS pickup_hour,
    COUNT(*)::INTEGER AS num_trips
  FROM read_parquet($clean)
  GROUP BY 1,2
)
SELECT * FROM agg ORDER BY PULocationID, pickup_hour
"""

df_zone_hour = con.execute(q_zone_hour, {"clean": CLEAN_PATH}).fetch_df()
df_zone_hour.to_parquet(ZONE_HOUR_PATH, index=False)
print(f"Aggregated zone×hour file saved: {ZONE_HOUR_PATH}")
df_zone_hour.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Aggregated zone×hour file saved: /content/processed/zone_hour_2024.parquet


,PULocationID,pickup_hour,num_trips
0,1,2024-01-25 06:00:00,1
1,1,2024-02-16 05:00:00,1
2,1,2024-04-15 05:00:00,1
3,1,2024-05-05 06:00:00,1
4,1,2024-07-14 15:00:00,1


In [25]:
import pandas as pd

df = pd.read_parquet(ZONE_HOUR_PATH).sort_values(["PULocationID","pickup_hour"])
df["lag_1h"]  = df.groupby("PULocationID")["num_trips"].shift(1)
df["lag_24h"] = df.groupby("PULocationID")["num_trips"].shift(24)
df = df.dropna(subset=["lag_1h","lag_24h"]).reset_index(drop=True)

LAGS_PATH = f"{OUT_DIR}/zone_hour_2024_with_lags.parquet"
df.to_parquet(LAGS_PATH, index=False)
print(f"Added lag features and saved: {LAGS_PATH}")
df.head()


Added lag features and saved: /content/processed/zone_hour_2024_with_lags.parquet


,PULocationID,pickup_hour,num_trips,lag_1h,lag_24h
0,2,2024-02-11 14:00:00,1,1.0,1.0
1,2,2024-02-12 15:00:00,1,1.0,1.0
2,2,2024-02-13 01:00:00,1,1.0,1.0
3,2,2024-02-14 17:00:00,1,1.0,1.0
4,2,2024-02-16 21:00:00,1,1.0,1.0


In [27]:
df_clean.head()

,pickup_datetime,PULocationID,DOLocationID
0,2024-01-01 00:28:08,161,158
1,2024-01-01 00:12:53,137,79
2,2024-01-01 00:23:05,79,186
3,2024-01-01 00:41:04,234,148
4,2024-01-01 00:57:21,148,97
